# School of Thinkers — DP (person level data, database extraction 2026-08-23)

Author: Luís Anunciação

- Source of truth: `Data/Official/` (the website's general export is lossy and was discarded).
- Recoding follows `EP_codebook.csv`; numeric `_num` columns are created next to the original text.
- Default sample: the 3 real municipal schools (O Semeador, Futuro Brilhante, Marton Lucca). Records from Escola Modelo (the platform's demonstration school) and from the Secretaria de Educação are excluded at the end of cleaning.
- Single output: `ep.rds` with df_st (students), df_par (families), df_t (teachers), df_dy (dyads).
- After the output, the notebook builds the Relatório 2 tables (descriptive tables, school comparison and the CFA supplement). The Paper 1 analyses are in the DA notebook.

In [ ]:
# DP §1 - load the 4 CSVs (UTF-8 with BOM), tag the instrument, standardize metadata
pacman::p_load(tidyverse)

raw <- imap(c(students = "Official/EP_alunos_saude_emocional.csv",
              parents = "Official/EP_pais_saude_emocional_familia.csv",
              teachers_practice = "Official/EP_professores_pratica_pedagogica.csv",
              teachers_qol = "Official/EP_professores_qualidade_vida_saude_mental.csv"),
            \(f, inst) read_csv(file.path("../Data", f), show_col_types = FALSE) |>
              mutate(instrument = inst) |>
              select(-survey_title, -target_audience, -respondent_type))
map(raw, dim) |> as_tibble()

In [ ]:
# DP §2 - cleaning and recoding; builds df_st, df_par, df_t, df_dy

# clean_rows(d, name_col, birth_col): keeps _status == completed, removes test names and,
# if birth_col is given, impossible birth dates (year >= 2026); prints rows removed per rule
clean_rows <- function(d, name_col, birth_col = NULL) {
  n0 <- nrow(d)
  d1 <- filter(d, `_status` == "completed")
  d2 <- filter(d1, !str_detect(replace_na(.data[[name_col]], ""), regex("teste|ttyy|tr5rrt", ignore_case = TRUE)))
  d3 <- if (is.null(birth_col)) d2 else filter(d2, coalesce(year(as_date(.data[[birth_col]])), 0) < 2026)
  cat(sprintf("%-18s raw %4d | -draft %d | -test name %d | -birth>=2026 %d | final %d\n",
              d$instrument[1], n0, n0 - nrow(d1), nrow(d1) - nrow(d2), nrow(d2) - nrow(d3), nrow(d3)))
  d3
}

# recode_items(d, cols, map): creates <col>_num keeping the original text; map = named vector or function
recode_items <- function(d, cols, map) {
  f <- if (is.function(map)) map else \(v) unname(map[v])
  mutate(d, across(all_of(cols), \(v) f(v), .names = "{.col}_num"))
}

first_digit <- \(v) as.numeric(str_extract(v, "\\d"))  # anchor digit (emoji or parentheses)
likert_freq <- c(Nunca = 1, Raramente = 2, `Às vezes` = 3, Frequentemente = 4, Sempre = 5)
likert_freq_upper <- c(NUNCA = 1, RARAMENTE = 2, `ÀS VEZES` = 3, FREQUENTEMENTE = 4, SEMPRE = 5)
yes_no <- c(Sim = 1, `Não` = 0)

target_schools <- c("O SEMEADOR", "Futuro Brilhante", "Marton Lucca")
# keep_schools(d): keeps only the 3 real municipal schools (Escola Modelo = platform demo
# school; Secretaria de Educação is not a school); prints what was removed
keep_schools <- function(d, col = "school_name") {
  removed <- d |> filter(!(.data[[col]] %in% target_schools)) |> count(.data[[col]])
  if (nrow(removed) > 0) {cat("  removed (outside the 3 schools): "); cat(sprintf("%s = %d; ", removed[[1]], removed$n)); cat("\n")}
  filter(d, .data[[col]] %in% target_schools)
}

cat("Cleaning (rows removed per rule):\n")
df_st <- clean_rows(raw$students, "respondent_name") |>
  keep_schools() |>
  recode_items(paste0("behavior_matrix.", 1:11), first_digit) |>
  recode_items(paste0("yes_no_questions.q", 12:18), yes_no) |>
  recode_items("demographics.age", \(v) as.numeric(str_extract(v, "\\d+")))

# caregiver birth dates with year >= 2026 (24 cases) are typos, not test rows: kept
df_par <- clean_rows(raw$parents, "respondent_name") |>
  keep_schools() |>
  recode_items(paste0("behavior_matrix.", 1:11), first_digit) |>
  recode_items(paste0("yes_no_questions.q", 12:18), yes_no)

df_t <- clean_rows(raw$teachers_practice, "demographics.full_name", "demographics.birth_date") |>
  recode_items(paste0("matrix_1.", 5:14), likert_freq) |>
  recode_items(paste0("matrix_2.", 15:25), first_digit) |>  # 0 = 'cannot say' (outside the 1-4 continuum; treat as missing in analysis)
  recode_items("initial_questions.q_1", yes_no) |>
  recode_items("initial_questions.q_2", likert_freq) |>
  full_join(clean_rows(raw$teachers_qol, "demographics.full_name", "demographics.birth_date") |>
              mutate(school_name = coalesce(school_name, `demographics.school`)) |>
              recode_items(paste0("questions_matrix.", 1:49), likert_freq_upper),
            by = "respondent_id", suffix = c("_pratica", "_qualidade")) |>
  mutate(school_name = coalesce(school_name_pratica, school_name_qualidade),
         full_name = coalesce(`demographics.full_name_pratica`, `demographics.full_name_qualidade`),
         has_pratica = !is.na(response_id_pratica),
         has_qualidade = !is.na(response_id_qualidade)) |>
  keep_schools()

# respondent_id is the child's ID in both emotional health inventories -> dyad
df_dy <- inner_join(df_st, df_par, by = "respondent_id", suffix = c("_aluno", "_familia"))

In [ ]:
# DP §3 - data quality report: n per school, item missingness, duplicates
quality_report <- function(d, label, school = "school_name") {
  cat(sprintf("\n--- %s: n = %d\n", label, nrow(d)))
  print(count(d, school = .data[[school]], sort = TRUE))
  d |>
    summarise(across(ends_with("_num"), \(v) sum(is.na(v)))) |>
    pivot_longer(everything(), names_to = "item", values_to = "n_missing") |>
    filter(n_missing > 0) |>
    arrange(desc(n_missing)) |>
    (\(m) {cat(sprintf("items with missing: %d\n", nrow(m))); if (nrow(m) > 0) print(head(m, 10))})()
  cat(sprintf("duplicated respondent_id: %d\n", sum(duplicated(d$respondent_id))))
}

quality_report(df_st, "df_st")
quality_report(df_par, "df_par")
quality_report(df_t, "df_t (people; item missingness is structural for those who answered only 1 survey; 17 QoL respondents stopped mid inventory)")
cat(sprintf("\n--- df_dy: n = %d (student and family completed for the same child)\n", nrow(df_dy)))
print(count(df_dy, school_name_aluno, sort = TRUE))
cat(sprintf("teachers in both surveys: %d\n", sum(df_t$has_pratica & df_t$has_qualidade)))

In [ ]:
# DP §4 - validation against expected totals and single output
tribble(
  ~check, ~ok,
  "raw: students 748", nrow(raw$students) == 748,
  "raw: parents 915", nrow(raw$parents) == 915,
  "raw: practice 97", nrow(raw$teachers_practice) == 97,
  "raw: qol 122", nrow(raw$teachers_qol) == 122,
  "completed: students 745", sum(raw$students$`_status` == "completed") == 745,
  "completed: practice 96", sum(raw$teachers_practice$`_status` == "completed") == 96,
  "completed: qol 122", sum(raw$teachers_qol$`_status` == "completed") == 122,
  "raw student-parent overlap 410", length(intersect(raw$students$respondent_id, raw$parents$respondent_id)) == 410,
  "final: students 741", nrow(df_st) == 741,
  "final: parents 912", nrow(df_par) == 912,
  "final: teachers 111", nrow(df_t) == 111,
  "final: dyads 406", nrow(df_dy) == 406
) |>
  (\(v) {print(as.data.frame(v)); stopifnot(all(v$ok)); v})() |>
  invisible()

cat(sprintf("completed: parents = %d\n", sum(raw$parents$`_status` == "completed")))

saveRDS(list(df_st = df_st, df_par = df_par,
             df_t = df_t, df_dy = df_dy), "ep.rds")
cat("Saved: ep.rds\n")

## Relatório 2 tables

In [ ]:
# Report setup: codebook, enrollment list (Table 1 denominator) and the formatters used below
pacman::p_load(readxl, lavaan, effectsize, broom)
cb <- read_csv("../Data/Official/EP_codebook.csv", show_col_types = FALSE)
roster <- read_xlsx("../Data/Old/Raw website export 2026-08-23/exportacao_geral_2026-08-23.xlsx", skip = 2,
                    col_names = paste0("c", 1:147), col_types = "text") |>
  select(escola = c1, turma = c2)

# formatters: M (SD), % yes, n (%)
msd <- \(v) sprintf("%.2f (%.2f)", mean(v, na.rm = TRUE), sd(v, na.rm = TRUE))
psim <- \(v) sprintf("%.1f%%", 100 * mean(v, na.rm = TRUE))
npct <- \(n, d) sprintf("%d (%.1f%%)", n, 100 * n / d)

stopifnot(nrow(roster) == 1694)

### Table 1 — Responses per school

Coverage of each respondent group against enrolment; sets the denominator for every percentage reported below.

In [ ]:
# Table 1 - responses per school (denominator: enrolled students); dyads and triads
roster |>
  count(escola, name = "N") |>
  full_join(count(df_st, escola = school_name, name = "aluno"), by = "escola") |>
  full_join(count(df_par, escola = school_name, name = "familia"), by = "escola") |>
  full_join(count(df_t, escola = school_name, name = "professor"), by = "escola") |>
  bind_rows(tibble(escola = "Total", N = nrow(roster), aluno = nrow(df_st),
                   familia = nrow(df_par), professor = nrow(df_t))) |>
  mutate(across(c(aluno, familia, professor), \(x) replace_na(x, 0L)),
         `Student n (%)` = if_else(is.na(N), "—", npct(aluno, N)),
         `Family n (%)` = if_else(is.na(N), "—", npct(familia, N))) |>
  select(School = escola, `N enrolled` = N, `Student n (%)`, `Family n (%)`, `Teacher, n` = professor) |>
  print(n = Inf)

# dyads per school and triads (dyad in a class evaluated by a Practice teacher)
count(df_dy, school_name_aluno, name = "dyads") |> print()
df_dy |>
  filter(class_id_aluno %in% df_t$demographics.class_evaluated) |>
  count(school_name_aluno, name = "triads") |>
  (\(x) {print(x); cat("total triads:", sum(x$triads), "\n")})()

### Table 2 — Respondent demographics

Describes who answered in each group; source of manuscript Table 1.

In [ ]:
# Table 2 - respondent demographics (APA format: label row, indented categories)
# count_pct(data, var, label): header row + indented n (%) frequencies
count_pct <- function(data, var, label) data |>
  count(category = replace_na({{var}}, "Not informed")) |>
  transmute(Characteristic = paste0("  ", category), Value = npct(n, sum(n))) |>
  (\(x) bind_rows(tibble(Characteristic = label, Value = ""), x))()

bind_rows(
  tibble(Characteristic = sprintf("Responding students (n = %d)", nrow(df_st)), Value = ""),
  # age: M/SD over valid values (na.rm)
  df_st |> summarise(Value = msd(demographics.age_num)) |> mutate(Characteristic = "  Age in years, M (SD)"),
  df_st |> mutate(sexo = if_else(demographics.gender %in% c("Outros", "Prefiro não informar"),
                                          "Other / prefer not to say", demographics.gender)) |> count_pct(sexo, "Sex"),
  df_st |> count_pct(demographics.household_size, "People at home (excluding the student)"),
  tibble(Characteristic = sprintf("Responding families (n = %d)", nrow(df_par)), Value = ""),
  df_par |> count_pct(family_structure.family_income, "Monthly family income"),
  df_par |> count_pct(family_structure.household_size, "People at home (excluding the respondent)"),
  df_par |> count_pct(child_development.premature, "Premature birth of the child"),
  df_par |> mutate(diag = if_else(str_detect(replace_na(child_development.diagnoses, ""),
                                                    "DI|TDAH|Dislexia|Discalculia|TEA"),
                                         "Any diagnosis", "No diagnosis")) |> count_pct(diag, "Child diagnosis"),
  tibble(Characteristic = sprintf("Responding teachers (n = %d)", nrow(df_t)), Value = ""),
  df_t |> mutate(tempo = coalesce(demographics.time_in_education_pratica,
                                            demographics.time_in_education_qualidade)) |>
    count_pct(tempo, "Years working in education")
) |> print(n = Inf)

### Table 3 — Matched items, all respondents

Item level means for students and families on the same content, before restricting to linked pairs.

In [ ]:
# Table 3 - parallel items, student x family, all respondents
# item_summary: M (SD) for scale items; % Yes for the additional 0/1 questions
item_summary <- \(d, col, sec) if (sec == "behavior_matrix") msd(d[[col]]) else psim(d[[col]])

itens <- cb |>
  filter(survey_title == "Inventário de Saúde Emocional - Aluno",
         section_id %in% c("behavior_matrix", "yes_no_questions"), type %in% c("matrix", "select")) |>
  select(column_name, label, section_id)

# student and family items are complete: no analysis below loses cases
stopifnot(!anyNA(df_st[paste0(itens$column_name, "_num")]),
          !anyNA(df_par[paste0(itens$column_name, "_num")]))

itens |>
  rowwise() |>
  mutate(Student = item_summary(df_st, paste0(column_name, "_num"), section_id),
         Family = item_summary(df_par, paste0(column_name, "_num"), section_id)) |>
  ungroup() |>
  select(Item = label, Student, Family) |>
  print(n = Inf)

### Table 4 — Matched items in the dyads

Same items in the subsample where child and caregiver are linked, so the two reports refer to the same child.

In [ ]:
# Table 4 - parallel items in the dyads (same child, same caregiver)
itens |>
  rowwise() |>
  mutate(Student = item_summary(df_dy, paste0(column_name, "_num_aluno"), section_id),
         Family = item_summary(df_dy, paste0(column_name, "_num_familia"), section_id)) |>
  ungroup() |>
  select(Item = label, Student, Family) |>
  print(n = Inf)

### Tables 5 and 6 — Teacher instrument items

Item level description of the two teacher inventories, reported separately because each has its own respondent set.

In [ ]:
# Tables 5 and 6 - teacher instrument items
# In Practice Part 2 (matrix_2), '(0) cannot say' lies outside the 1-4 continuum: becomes NA
teacher_items <- function(d, survey, sections, zero_na = FALSE) cb |>
  filter(survey_title == survey, section_id %in% sections, type == "matrix") |>
  rowwise() |>
  mutate(v = list(na_if(d[[paste0(column_name, "_num")]] * 1, if (zero_na && section_id == "matrix_2") 0 else -1)),
         n = sum(!is.na(v)), `M (SD)` = msd(v)) |>
  ungroup() |>
  select(Item = label, Dimension = dimension, n, `M (SD)`)

teacher_items(filter(df_t, has_pratica), "INVENTÁRIO DE PRÁTICA PEDAGÓGICA E PERFIL DA TURMA",
           c("matrix_1", "matrix_2"), zero_na = TRUE) |>
  mutate(Dimension = if_else(str_detect(Dimension, "Não sei informar"), "", Dimension)) |>
  print(n = Inf)
teacher_items(filter(df_t, has_qualidade), "Inventário de Qualidade de Vida e Saúde Mental",
           "questions_matrix") |>
  print(n = Inf)

### Table 7 — School comparison

Tests whether the three schools differ on each dimension before results are pooled.

In [ ]:
# Table 7 - school comparison: score = mean of the dimension's valid items (na.rm)
# Codebook dimensions assumed correct; the three frames already contain only the three schools
scores <- bind_rows(
  df_st |> mutate(Group = "Students", survey_title = "Inventário de Saúde Emocional - Aluno"),
  df_par |> mutate(Group = "Families", survey_title = "Inventário de Saúde Emocional - Família"),
  df_t |> filter(has_qualidade) |>
    mutate(Group = "Teachers (QoL)", survey_title = "Inventário de Qualidade de Vida e Saúde Mental")) |>
  pivot_longer(matches("^(behavior_matrix|questions_matrix)\\.\\d+_num$"),
               names_to = "column_name", values_to = "v", values_drop_na = TRUE) |>
  inner_join(cb |> filter(type == "matrix") |>
               transmute(survey_title, column_name = paste0(column_name, "_num"), dimension),
             by = c("survey_title", "column_name")) |>
  summarise(score = mean(v), .by = c(Group, dimension, school_name, respondent_id))

scores |>
  summarise(ms = msd(score), .by = c(Group, dimension, school_name)) |>
  pivot_wider(names_from = school_name, values_from = ms) |>
  left_join(scores |>
              nest(.by = c(Group, dimension)) |>
              mutate(fit = map(data, \(d) aov(score ~ school_name, d)),
                     eta2 = map_dbl(fit, \(m) eta_squared(m, partial = FALSE, ci = NULL)$Eta2),
                     stats = map(fit, \(m) tidy(m)[1, c("statistic", "p.value")])) |>
              unnest(stats) |>
              transmute(Group, dimension, F = sprintf("%.2f", statistic),
                        p = sprintf("%.3f", p.value), eta2 = sprintf("%.3f", eta2)),
            by = c("Group", "dimension")) |>
  rename(Dimension = dimension) |>
  print(n = Inf, width = Inf)  # eta2 column would be elided at the default width

### Tables S1 and S2 — CFA of the codebook structure

Tests the structure the instrument was designed with; baseline against which the two factor model of Part B is judged.

In [ ]:
# Tables S1 and S2 - CFA of the structure proposed in the codebook (dimensions assumed correct)
# Model: one factor per codebook dimension, items renamed i<item_id>
# cfa_items(survey, section): codebook items of a matrix section (model name, data column, label, dimension)
cfa_items <- \(survey, section) cb |>
  filter(survey_title == survey, section_id == section, type == "matrix") |>
  transmute(vs = paste0("i", item_id), col = paste0(column_name, "_num"), label, dimension)
# cfa_syntax(items): lavaan model string, one factor per dimension
cfa_syntax <- \(items) items |>
  summarise(eq = paste(make.names(first(dimension)), "=~", paste(vs, collapse = " + ")), .by = dimension) |>
  pull(eq) |> paste(collapse = "\n")

it_st <- cfa_items("Inventário de Saúde Emocional - Aluno", "behavior_matrix")
it_par <- cfa_items("Inventário de Saúde Emocional - Família", "behavior_matrix")
it_t <- cfa_items("Inventário de Qualidade de Vida e Saúde Mental", "questions_matrix")
m_st <- cfa_syntax(it_st); m_par <- cfa_syntax(it_par); m_t <- cfa_syntax(it_t)
cat(m_st, m_par, m_t, sep = "\n\n")

In [ ]:
# Run: WLSMV (ordinal indicators), listwise complete cases. Student and family items are complete;
# QoL respondents who stopped mid inventory are dropped (n reported in the table label)
# cfa_data(d, items): section items renamed to the model names, complete cases only
cfa_data <- \(d, items) d |> select(all_of(set_names(items$col, items$vs))) |> drop_na()
fit_st <- cfa(m_st, cfa_data(df_st, it_st), ordered = TRUE, estimator = "WLSMV")
fit_par <- cfa(m_par, cfa_data(df_par, it_par), ordered = TRUE, estimator = "WLSMV")
fit_t <- cfa(m_t, cfa_data(filter(df_t, has_qualidade), it_t), ordered = TRUE, estimator = "WLSMV")

In [ ]:
# Summary: Table S1 (scaled fit indices) and Table S2 (standardized loadings)
# cfa_tables(fit, items, model): list(fit_row, loadings) for one fitted model
cfa_tables <- \(fit, items, model) list(
  fit_row = fitMeasures(fit, c("chisq.scaled", "df", "cfi.scaled", "tli.scaled",
                               "rmsea.scaled", "rmsea.ci.lower.scaled", "rmsea.ci.upper.scaled", "srmr")) |>
    (\(fm) tibble(Model = sprintf("%s (n = %d)", model, lavInspect(fit, "nobs")),
                  chi2_df = sprintf("%.1f (%d)", fm[1], fm[2]), CFI = sprintf("%.3f", fm[3]),
                  TLI = sprintf("%.3f", fm[4]), RMSEA = sprintf("%.3f [%.3f, %.3f]", fm[5], fm[6], fm[7]),
                  SRMR = sprintf("%.3f", fm[8])))(),
  loadings = standardizedSolution(fit) |> filter(op == "=~") |>
    left_join(select(items, vs, label, dimension), by = c("rhs" = "vs")) |>
    transmute(Model = sprintf("%s (n = %d)", model, lavInspect(fit, "nobs")), Factor = dimension,
              Item = label, Loading = sprintf("%.2f", est.std)))

models <- list(cfa_tables(fit_st, it_st, "Students: self efficacy, 3 factors"),
               cfa_tables(fit_par, it_par, "Families: self efficacy, 3 factors"),
               cfa_tables(fit_t, it_t, "Teachers QoL: 6 factors"))
map_dfr(models, "fit_row") |> as.data.frame() |> print()
map_dfr(models, "loadings") |> as.data.frame() |> print()

### Table S3 — Enrolled and respondents per school and class

Class level breakdown of Table 1, including the Practice teachers linked to each class.

In [ ]:
# Table S3 - enrolled and respondents per school and class; Practice teachers
# linked through 'class evaluated' (class_evaluated = students'/families' class_id)
bind_rows(mutate(roster, turma = "(all classes)"), roster) |>
  count(escola, turma, name = "N") |>
  left_join(bind_rows(mutate(df_st, class_name = "(all classes)"), df_st) |>
              count(escola = school_name, turma = class_name, name = "aluno"), by = c("escola", "turma")) |>
  left_join(bind_rows(mutate(df_par, class_name = "(all classes)"), df_par) |>
              count(escola = school_name, turma = class_name, name = "familia"), by = c("escola", "turma")) |>
  left_join(df_t |> filter(has_pratica) |>
              inner_join(distinct(bind_rows(df_st, df_par), class_id, escola = school_name, turma = class_name),
                         by = c("demographics.class_evaluated" = "class_id")) |>
              count(escola, turma, name = "prof_pratica"), by = c("escola", "turma")) |>
  arrange(escola, turma != "(all classes)", turma) |>
  mutate(across(c(aluno, familia, prof_pratica), \(x) replace_na(x, 0L)),
         `Student n (%)` = npct(aluno, N), `Family n (%)` = npct(familia, N)) |>
  select(School = escola, Class = turma, `N enrolled` = N, `Student n (%)`, `Family n (%)`,
         `Teacher (Practice), n` = prof_pratica) |>
  print(n = Inf)